In [1]:
import numpy as np
import pandas as pd
import gurobipy as gp
from gurobipy import GRB
from scipy.stats import norm

In [2]:
alpha = 0.95
k = norm.pdf(norm.ppf(alpha)) / (1 - alpha)
print(k)

2.0627128075074257


In [3]:
df = pd.read_csv("selected_monthly_return.csv", encoding="utf-8-sig")
tickers = df["Ticker"].tolist()
returns = df.drop(columns=["PERMNO", "Ticker", "average_monthly_return"])
mu = df["average_monthly_return"].to_numpy(dtype=float)

In [4]:
cov_df = pd.read_csv(
    "covariance_matrix.csv",
    skiprows=1,
    encoding="utf-8-sig"
)

Sigma = cov_df.drop(
    columns=["PERMNO", "Ticker"]
).to_numpy(dtype=float)

eigvals, eigvecs = np.linalg.eigh(Sigma)

A = np.sqrt(np.clip(eigvals, 0, None))[:, None] * eigvecs.T

In [5]:
sp500 = pd.read_csv(
    "SP500_monthly_202001_202603.csv",
    encoding="utf-8-sig"
)

sp500["Month"] = pd.to_datetime(sp500["Month"])

b = sp500.loc[
    sp500["Month"].between("2021-01-01", "2025-12-31"),
    "MonthlyReturn"
].mean()

In [6]:
m = gp.Model("portfolio")

w = m.addMVar(
    len(tickers),
    vtype=GRB.CONTINUOUS,
    name="w"
)

s = m.addVar(
    vtype=GRB.CONTINUOUS,
    name="s"
)

Restricted license - for non-production use only - expires 2027-11-29


In [7]:
m.addConstr(w.sum() == 1, name="full_investment")
m.addConstr(w <= 0.2, name="max_weight")
m.addConstr(w @ mu >= b, name="minimum_return")

<MConstr () *awaiting model update*>

In [8]:
y = m.addMVar(
    37,
    lb=-GRB.INFINITY,
    name="y"
)

m.addConstr(y == A @ w)


m.addGenConstrNorm(s, y, 2.0)

<gurobi.GenConstr *Awaiting Model Update*>

In [9]:
m.setObjective(
    -1 * w @ mu + k * s,
    gp.GRB.MINIMIZE
)

In [10]:
m.optimize()

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: AMD Ryzen 9 8945HX with Radeon Graphics, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 16 physical cores, 32 logical processors, using up to 32 threads

Optimize a model with 76 rows, 75 columns and 1517 nonzeros (Min)
Model fingerprint: 0xe4086c47
Model has 38 linear objective coefficients
Model has 1 simple general constraint
  1 NORM
Variable types: 75 continuous, 0 integer (0 binary)
Coefficient statistics:
  Matrix range     [1e-06, 1e+00]
  Objective range  [2e-03, 2e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e-02, 1e+00]

Presolve removed 37 rows and 0 columns
Presolve time: 0.01s
Presolved: 40 rows, 112 columns, 1512 nonzeros
Presolved model has 37 quadratic constraint(s)
Variable types: 112 continuous, 0 integer (0 binary)
Root relaxation presolve time: 0.01s
Root relaxation presolved: 114 rows, 186 columns, 1734 nonzeros
Root relaxation presolved model has 37

In [11]:
if m.Status == GRB.OPTIMAL:

    result = pd.DataFrame({
        "Ticker": tickers,
        "Weight": w.X
    })

    print("\nOptimal portfolio weights:")

    print(
        result.to_string(
            index=False,
            formatters={"Weight": "{:.6%}".format}
        )
    )

    print(f"\nSum of weights: {w.X.sum():.6f}")
    print(f"Benchmark mean monthly return: {b:.6%}")
    print(f"Portfolio expected monthly return: {mu @ w.X:.6%}")
    print(f"Portfolio monthly standard deviation: {s.X:.6%}")
    print(f"Portfolio monthly 95% CVaR: {m.ObjVal:.6%}")

else:
    print(f"Optimization ended with status {m.Status}")


Optimal portfolio weights:
Ticker     Weight
  COHR  0.000003%
    GE  0.000246%
   FNV  0.001858%
    UI  0.000135%
  BABA  3.747166%
  LITE  0.000004%
  CVNA  0.000003%
   CVS  6.797882%
   NET  0.000003%
  PLTR  0.000002%
  SOFI  0.000002%
   NEM  0.000047%
   CAH 19.983605%
   GLW  0.437228%
   HWM  0.000021%
  KLAC  0.000052%
  LRCX  0.000011%
    MU  0.000025%
  INTC  0.000013%
   AMD  0.000011%
   WDC  0.000009%
     B  0.000126%
  IDXX  0.000003%
   AEM  0.000010%
  ESLT 19.997301%
  CIEN  0.000052%
   APH  0.000383%
   FIX  7.680201%
    MT  0.000003%
   CLS  0.000058%
  INSM  4.130385%
    DB  8.013164%
   STX  0.000023%
   KGC  0.000098%
   NRG  0.000204%
   WPM 14.003785%
  AVGO 15.205885%

Sum of weights: 1.000000
Benchmark mean monthly return: 1.099027%
Portfolio expected monthly return: 2.997852%
Portfolio monthly standard deviation: 4.328196%
Portfolio monthly 95% CVaR: 5.929974%
